<a href="https://colab.research.google.com/github/aycaaozturk/AML-project/blob/main/AML_XGBoost_cross_val.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q xgboost scikit-survival

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 13.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 115.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 148.0 MB/s eta 0:00:00


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.model_selection import StratifiedKFold

from scipy.stats import (
    mannwhitneyu,
    kruskal
)

from sksurv.metrics import (
    concordance_index_censored,
    brier_score
)

from sksurv.nonparametric import (
    kaplan_meier_estimator
)

In [ ]:
BASE_DIR = Path(
    "/content/drive/MyDrive/Uniklinikum Würzburg/AML/"
    "Output Files 2/combined_survival_model_ready_wout_protocol/"
)

TRAIN_PATH = (
    BASE_DIR
    / "combined_train_survival_model_ready.csv"
)

VALIDATION_PATH = (
    BASE_DIR
    / "combined_validation_survival_model_ready.csv"
)

TEST_PATH = (
    BASE_DIR
    / "combined_test_survival_model_ready.csv"
)

RAW_CLINICAL_PATH = (
    "/content/drive/MyDrive/Uniklinikum Würzburg/AML/"
    "AML Dataset/aml_target_2018_pub/"
    "data_clinical_patient.txt"
)

OUTPUT_DIR = (
    BASE_DIR
    / "xgboost_fairness_cv_sensitivity"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(OUTPUT_DIR)

/content/drive/MyDrive/Uniklinikum Würzburg/AML/Output Files 2/combined_survival_model_ready_wout_protocol/xgboost_fairness_cv_sensitivity


In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VALIDATION_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (620, 68)
Validation: (177, 68)
Test: (89, 68)


In [ ]:
development_df = pd.concat(
    [
        train_df,
        val_df
    ],
    ignore_index=True
)

print(
    "Development dataset:",
    development_df.shape
)

print(
    "Independent test set:",
    test_df.shape
)

Development dataset: (797, 68)
Independent test set: (89, 68)


In [ ]:
DURATION_COL = "OS_DAYS"
EVENT_COL = "OS_EVENT"

PATIENT_ID_COL = "PATIENT_ID"
SAMPLE_ID_COL = "SAMPLE_ID"

EXCLUDE_COLS = [
    PATIENT_ID_COL,
    SAMPLE_ID_COL,
    DURATION_COL,
    EVENT_COL
]

In [ ]:
ORIGINAL_FEATURES = [
    col
    for col in development_df.columns
    if col not in EXCLUDE_COLS
]

ORIGINAL_FEATURES = [
    col
    for col in ORIGINAL_FEATURES
    if pd.api.types.is_numeric_dtype(
        development_df[col]
    )
]

zero_variance_features = [
    feature
    for feature in ORIGINAL_FEATURES
    if development_df[
        feature
    ].nunique(
        dropna=True
    ) <= 1
]

ORIGINAL_FEATURES = [
    feature
    for feature in ORIGINAL_FEATURES
    if feature not in zero_variance_features
]

print(
    "Original feature count:",
    len(ORIGINAL_FEATURES)
)

Original feature count: 64


In [ ]:
SENSITIVE_FEATURES = [
    feature
    for feature in ORIGINAL_FEATURES
    if (
        feature == "AGE"
        or feature.startswith("SEX_")
        or feature.startswith("RACE_")
    )
]

print(
    "Removed features:"
)

for feature in SENSITIVE_FEATURES:
    print("-", feature)

Removed features:
- AGE
- SEX_Female
- SEX_Male
- RACE_American Indian or Alaska Native
- RACE_Asian
- RACE_Black or African American
- RACE_Native Hawaiian or other Pacific Islander
- RACE_Other
- RACE_White


In [ ]:
SENSITIVITY_FEATURES = [
    feature
    for feature in ORIGINAL_FEATURES
    if feature not in SENSITIVE_FEATURES
]

print(
    "Original:",
    len(ORIGINAL_FEATURES)
)

print(
    "Without A/S/R:",
    len(SENSITIVITY_FEATURES)
)

Original: 64
Without A/S/R: 55


In [ ]:
raw_df = pd.read_csv(
    RAW_CLINICAL_PATH,
    sep="\t",
    comment="#"
)

demographic_lookup = (
    raw_df[
        [
            "PATIENT_ID",
            "AGE",
            "SEX",
            "RACE"
        ]
    ]
    .drop_duplicates(
        "PATIENT_ID"
    )
    .rename(
        columns={
            "AGE":
                "AGE_ORIGINAL",
            "SEX":
                "SEX_GROUP",
            "RACE":
                "RACE_GROUP"
        }
    )
)

In [ ]:
development_df = (
    development_df.merge(
        demographic_lookup,
        on="PATIENT_ID",
        how="left",
        validate="one_to_one"
    )
)

print(
    "Missing age:",
    development_df[
        "AGE_ORIGINAL"
    ].isna().sum()
)

print(
    "Missing sex:",
    development_df[
        "SEX_GROUP"
    ].isna().sum()
)

print(
    "Missing race:",
    development_df[
        "RACE_GROUP"
    ].isna().sum()
)

Missing age: 0
Missing sex: 0
Missing race: 0


In [ ]:
development_df[
    "AGE_GROUP"
] = pd.qcut(
    development_df[
        "AGE_ORIGINAL"
    ],
    q=3,
    labels=[
        "Younger",
        "Middle",
        "Older"
    ],
    duplicates="drop"
)

display(
    development_df[
        "AGE_GROUP"
    ]
    .value_counts()
    .sort_index()
)

,count
AGE_GROUP,
Younger,271
Middle,288
Older,238


In [ ]:
BEST_PARAMS = {
    "objective": "survival:cox",
    "eval_metric": "cox-nloglik",

    "tree_method": "hist",

    "max_depth": 5,
    "eta": 0.03,
    "min_child_weight": 3,

    "subsample": 0.70,
    "colsample_bytree": 0.70,

    "lambda": 5.0,
    "alpha": 0.10,

    "seed": 42,
    "verbosity": 0
}

In [ ]:
def make_xgb_survival_labels(
    df
):

    durations = (
        df[
            DURATION_COL
        ]
        .astype(float)
        .values
    )

    events = (
        df[
            EVENT_COL
        ]
        .astype(int)
        .values
    )

    labels = np.where(
        events == 1,
        durations,
        -durations
    )

    return labels

In [ ]:
def calculate_cindex(
    events,
    durations,
    risk
):

    try:

        return (
            concordance_index_censored(
                events.astype(bool),
                durations.astype(float),
                risk
            )[0]
        )

    except Exception:

        return np.nan

In [ ]:
def show_stratum_counts(
    df,
    group_column
):

    table = (
        df.groupby(
            [
                EVENT_COL,
                group_column
            ],
            observed=True
        )
        .size()
        .reset_index(
            name="N"
        )
    )

    print(
        f"\nOutcome + {group_column}"
    )

    display(table)


show_stratum_counts(
    development_df,
    "SEX_GROUP"
)

show_stratum_counts(
    development_df,
    "AGE_GROUP"
)

show_stratum_counts(
    development_df,
    "RACE_GROUP"
)


Outcome + SEX_GROUP


,OS_EVENT,SEX_GROUP,N
0,0.0,Female,239
1,0.0,Male,272
2,1.0,Female,146
3,1.0,Male,140



Outcome + AGE_GROUP


,OS_EVENT,AGE_GROUP,N
0,0.0,Younger,178
1,0.0,Middle,192
2,0.0,Older,141
3,1.0,Younger,93
4,1.0,Middle,96
5,1.0,Older,97



Outcome + RACE_GROUP


,OS_EVENT,RACE_GROUP,N
0,0.0,American Indian or Alaska Native,3
1,0.0,Asian,19
2,0.0,Black or African American,42
3,0.0,Native Hawaiian or other Pacific Islander,3
4,0.0,Other,23
5,0.0,Unknown,27
6,0.0,White,394
7,1.0,American Indian or Alaska Native,1
8,1.0,Asian,20
9,1.0,Black or African American,51


In [ ]:
def create_stratification_labels(
    df,
    group_column,
    requested_splits=5
):

    combined_labels = (
        df[
            EVENT_COL
        ]
        .astype(str)
        +
        "_"
        +
        df[
            group_column
        ]
        .astype(str)
    )

    counts = (
        combined_labels
        .value_counts()
    )

    min_combined_count = (
        counts.min()
    )

    if (
        min_combined_count
        >= requested_splits
    ):

        print(
            f"{group_column}: "
            f"{requested_splits}-fold "
            f"Outcome + subgroup stratification."
        )

        return (
            combined_labels,
            requested_splits,
            "Outcome + subgroup"
        )

    if min_combined_count >= 3:

        n_splits = min(
            requested_splits,
            int(
                min_combined_count
            )
        )

        print(
            f"{group_column}: "
            f"{n_splits}-fold "
            f"Outcome + subgroup stratification."
        )

        return (
            combined_labels,
            n_splits,
            "Outcome + subgroup"
        )

    outcome_labels = (
        df[
            EVENT_COL
        ]
        .astype(str)
    )

    outcome_counts = (
        outcome_labels
        .value_counts()
    )

    n_splits = min(
        requested_splits,
        int(
            outcome_counts.min()
        )
    )

    print(
        f"{group_column}: "
        f"Outcome + subgroup not feasible."
    )

    print(
        f"Falling back to "
        f"{n_splits}-fold outcome-only stratification."
    )

    return (
        outcome_labels,
        n_splits,
        "Outcome only"
    )

In [ ]:
def estimate_breslow_baseline_hazard(
    durations,
    events,
    relative_hazards
):

    durations = np.asarray(
        durations,
        dtype=float
    )

    events = np.asarray(
        events,
        dtype=int
    )

    relative_hazards = np.asarray(
        relative_hazards,
        dtype=float
    )

    event_times = np.sort(
        np.unique(
            durations[
                events == 1
            ]
        )
    )

    increments = []

    for t in event_times:

        d_t = np.sum(
            (durations == t)
            &
            (events == 1)
        )

        risk_set = (
            durations >= t
        )

        denominator = np.sum(
            relative_hazards[
                risk_set
            ]
        )

        increments.append(
            d_t
            /
            denominator
        )

    cumulative_hazard = (
        np.cumsum(
            increments
        )
    )

    return (
        event_times,
        cumulative_hazard
    )

In [ ]:
def get_baseline_cumulative_hazard_at_time(
    t,
    baseline_times,
    baseline_cumhaz
):

    mask = (
        baseline_times <= t
    )

    if not np.any(mask):
        return 0.0

    return (
        baseline_cumhaz[
            mask
        ][-1]
    )

In [ ]:
def survival_probability(
    raw_margin,
    baseline_cumulative_hazard
):

    relative_hazard = np.exp(
        raw_margin
    )

    return np.exp(
        -baseline_cumulative_hazard
        *
        relative_hazard
    )

In [ ]:
def km_survival_at_time(
    events,
    durations,
    horizon
):

    times, survival = (
        kaplan_meier_estimator(
            events.astype(bool),
            durations.astype(float)
        )
    )

    valid = (
        times <= horizon
    )

    if not np.any(valid):
        return 1.0

    return survival[
        valid
    ][-1]

In [ ]:
HORIZONS = {
    "1Y": 365,
    "2Y": 730,
    "3Y": 1095
}

In [ ]:
def run_xgb_fairness_cv(
    df,
    group_column,
    original_features,
    sensitivity_features,
    requested_splits=5,
    random_state=42
):

    strat_labels, n_splits, strat_method = (
        create_stratification_labels(
            df,
            group_column,
            requested_splits
        )
    )

    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    overall_results = []
    subgroup_results = []
    brier_results = []
    calibration_results = []

    for fold, (
        train_idx,
        val_idx
    ) in enumerate(
        cv.split(
            df,
            strat_labels
        ),
        start=1
    ):

        print(
            f"\n{group_column} "
            f"Fold {fold}/{n_splits}"
        )

        fold_train = (
            df.iloc[
                train_idx
            ]
            .copy()
        )

        fold_val = (
            df.iloc[
                val_idx
            ]
            .copy()
        )

        # ====================================================
        # X matrices
        # ====================================================

        X_train_original = (
            fold_train[
                original_features
            ]
            .astype(float)
        )

        X_val_original = (
            fold_val[
                original_features
            ]
            .astype(float)
        )

        X_train_sensitivity = (
            fold_train[
                sensitivity_features
            ]
            .astype(float)
        )

        X_val_sensitivity = (
            fold_val[
                sensitivity_features
            ]
            .astype(float)
        )

        # ====================================================
        # Cox labels
        # ====================================================

        y_train_xgb = (
            make_xgb_survival_labels(
                fold_train
            )
        )

        y_val_xgb = (
            make_xgb_survival_labels(
                fold_val
            )
        )

        # ====================================================
        # DMatrix
        # ====================================================

        dtrain_original = xgb.DMatrix(
            X_train_original,
            label=y_train_xgb,
            feature_names=original_features
        )

        dval_original = xgb.DMatrix(
            X_val_original,
            label=y_val_xgb,
            feature_names=original_features
        )

        dtrain_sensitivity = xgb.DMatrix(
            X_train_sensitivity,
            label=y_train_xgb,
            feature_names=sensitivity_features
        )

        dval_sensitivity = xgb.DMatrix(
            X_val_sensitivity,
            label=y_val_xgb,
            feature_names=sensitivity_features
        )

        # ====================================================
        # Train original model
        # ====================================================

        model_original = xgb.train(
            params=BEST_PARAMS,
            dtrain=dtrain_original,
            num_boost_round=2000,
            evals=[
                (
                    dtrain_original,
                    "train"
                ),
                (
                    dval_original,
                    "validation"
                )
            ],
            early_stopping_rounds=50,
            verbose_eval=False
        )

        best_iter_original = (
            model_original.best_iteration
            + 1
        )

        # ====================================================
        # Train sensitivity model
        # ====================================================

        model_sensitivity = xgb.train(
            params=BEST_PARAMS,
            dtrain=dtrain_sensitivity,
            num_boost_round=2000,
            evals=[
                (
                    dtrain_sensitivity,
                    "train"
                ),
                (
                    dval_sensitivity,
                    "validation"
                )
            ],
            early_stopping_rounds=50,
            verbose_eval=False
        )

        best_iter_sensitivity = (
            model_sensitivity.best_iteration
            + 1
        )

        # ====================================================
        # Raw margins
        # ====================================================

        risk_original = (
            model_original.predict(
                dval_original,
                output_margin=True,
                iteration_range=(
                    0,
                    best_iter_original
                )
            )
        )

        risk_sensitivity = (
            model_sensitivity.predict(
                dval_sensitivity,
                output_margin=True,
                iteration_range=(
                    0,
                    best_iter_sensitivity
                )
            )
        )

        fold_val[
            "RISK_ORIGINAL"
        ] = risk_original

        fold_val[
            "RISK_WITHOUT_ASR"
        ] = risk_sensitivity

        # ====================================================
        # Overall C-index
        # ====================================================

        c_orig = calculate_cindex(
            fold_val[
                EVENT_COL
            ].values,
            fold_val[
                DURATION_COL
            ].values,
            risk_original
        )

        c_sens = calculate_cindex(
            fold_val[
                EVENT_COL
            ].values,
            fold_val[
                DURATION_COL
            ].values,
            risk_sensitivity
        )

        overall_results.append({
            "Fold":
                fold,

            "Stratification":
                strat_method,

            "Original_Cindex":
                c_orig,

            "Without_ASR_Cindex":
                c_sens,

            "Change":
                c_sens - c_orig,

            "Best_Iter_Original":
                best_iter_original,

            "Best_Iter_Without_ASR":
                best_iter_sensitivity
        })

        # ====================================================
        # Training margins for Breslow
        # ====================================================

        train_margin_original = (
            model_original.predict(
                dtrain_original,
                output_margin=True,
                iteration_range=(
                    0,
                    best_iter_original
                )
            )
        )

        train_margin_sensitivity = (
            model_sensitivity.predict(
                dtrain_sensitivity,
                output_margin=True,
                iteration_range=(
                    0,
                    best_iter_sensitivity
                )
            )
        )

        durations_train_fold = (
            fold_train[
                DURATION_COL
            ]
            .astype(float)
            .values
        )

        events_train_fold = (
            fold_train[
                EVENT_COL
            ]
            .astype(int)
            .values
        )

        # ====================================================
        # Separate baseline hazards
        # ====================================================

        baseline_times_orig, baseline_cumhaz_orig = (
            estimate_breslow_baseline_hazard(
                durations_train_fold,
                events_train_fold,
                np.exp(
                    train_margin_original
                )
            )
        )

        baseline_times_sens, baseline_cumhaz_sens = (
            estimate_breslow_baseline_hazard(
                durations_train_fold,
                events_train_fold,
                np.exp(
                    train_margin_sensitivity
                )
            )
        )

        # ====================================================
        # Survival probabilities
        # ====================================================

        for label, horizon in (
            HORIZONS.items()
        ):

            H0_orig = (
                get_baseline_cumulative_hazard_at_time(
                    horizon,
                    baseline_times_orig,
                    baseline_cumhaz_orig
                )
            )

            H0_sens = (
                get_baseline_cumulative_hazard_at_time(
                    horizon,
                    baseline_times_sens,
                    baseline_cumhaz_sens
                )
            )

            fold_val[
                f"ORIGINAL_SURVIVAL_{label}"
            ] = survival_probability(
                risk_original,
                H0_orig
            )

            fold_val[
                f"WITHOUT_ASR_SURVIVAL_{label}"
            ] = survival_probability(
                risk_sensitivity,
                H0_sens
            )

        # ====================================================
        # Structured training survival data for Brier
        # ====================================================

        y_train_surv = np.array(
            list(
                zip(
                    fold_train[
                        EVENT_COL
                    ].astype(bool),

                    fold_train[
                        DURATION_COL
                    ].astype(float)
                )
            ),
            dtype=[
                ("event", "?"),
                ("time", "<f8")
            ]
        )

        # ====================================================
        # Subgroup analysis
        # ====================================================

        for group, subgroup in (
            fold_val.groupby(
                group_column,
                observed=True
            )
        ):

            n = len(subgroup)

            events = int(
                subgroup[
                    EVENT_COL
                ].sum()
            )

            # -----------------------------------------------
            # Subgroup C-index
            # -----------------------------------------------

            if (
                n >= 5
                and events >= 2
            ):

                c_group_orig = (
                    calculate_cindex(
                        subgroup[
                            EVENT_COL
                        ].values,

                        subgroup[
                            DURATION_COL
                        ].values,

                        subgroup[
                            "RISK_ORIGINAL"
                        ].values
                    )
                )

                c_group_sens = (
                    calculate_cindex(
                        subgroup[
                            EVENT_COL
                        ].values,

                        subgroup[
                            DURATION_COL
                        ].values,

                        subgroup[
                            "RISK_WITHOUT_ASR"
                        ].values
                    )
                )

            else:

                c_group_orig = np.nan
                c_group_sens = np.nan

            subgroup_results.append({
                "Fold":
                    fold,

                "Group":
                    group,

                "N":
                    n,

                "Events":
                    events,

                "Original_Cindex":
                    c_group_orig,

                "Without_ASR_Cindex":
                    c_group_sens,

                "Change":
                    c_group_sens
                    -
                    c_group_orig
            })

            # -----------------------------------------------
            # Brier
            # -----------------------------------------------

            if n >= 5:

                y_sub = np.array(
                    list(
                        zip(
                            subgroup[
                                EVENT_COL
                            ].astype(bool),

                            subgroup[
                                DURATION_COL
                            ].astype(float)
                        )
                    ),
                    dtype=[
                        ("event", "?"),
                        ("time", "<f8")
                    ]
                )

                probs_orig = subgroup[
                    [
                        "ORIGINAL_SURVIVAL_1Y",
                        "ORIGINAL_SURVIVAL_2Y",
                        "ORIGINAL_SURVIVAL_3Y"
                    ]
                ].to_numpy()

                probs_sens = subgroup[
                    [
                        "WITHOUT_ASR_SURVIVAL_1Y",
                        "WITHOUT_ASR_SURVIVAL_2Y",
                        "WITHOUT_ASR_SURVIVAL_3Y"
                    ]
                ].to_numpy()

                try:

                    _, bs_orig = brier_score(
                        y_train_surv,
                        y_sub,
                        probs_orig,
                        np.array(
                            [
                                365.,
                                730.,
                                1095.
                            ]
                        )
                    )

                    _, bs_sens = brier_score(
                        y_train_surv,
                        y_sub,
                        probs_sens,
                        np.array(
                            [
                                365.,
                                730.,
                                1095.
                            ]
                        )
                    )

                except Exception:

                    bs_orig = [
                        np.nan,
                        np.nan,
                        np.nan
                    ]

                    bs_sens = [
                        np.nan,
                        np.nan,
                        np.nan
                    ]

            else:

                bs_orig = [
                    np.nan,
                    np.nan,
                    np.nan
                ]

                bs_sens = [
                    np.nan,
                    np.nan,
                    np.nan
                ]

            brier_results.append({
                "Fold":
                    fold,

                "Group":
                    group,

                "N":
                    n,

                "Events":
                    events,

                "Original_BS_1Y":
                    bs_orig[0],

                "Without_ASR_BS_1Y":
                    bs_sens[0],

                "Original_BS_2Y":
                    bs_orig[1],

                "Without_ASR_BS_2Y":
                    bs_sens[1],

                "Original_BS_3Y":
                    bs_orig[2],

                "Without_ASR_BS_3Y":
                    bs_sens[2]
            })

            # -----------------------------------------------
            # Calibration absolute error
            # -----------------------------------------------

            cal_row = {
                "Fold":
                    fold,

                "Group":
                    group,

                "N":
                    n,

                "Events":
                    events
            }

            if n >= 5:

                for label, horizon in (
                    HORIZONS.items()
                ):

                    observed = (
                        km_survival_at_time(
                            subgroup[
                                EVENT_COL
                            ].values,

                            subgroup[
                                DURATION_COL
                            ].values,

                            horizon
                        )
                    )

                    pred_orig = subgroup[
                        f"ORIGINAL_SURVIVAL_{label}"
                    ].mean()

                    pred_sens = subgroup[
                        f"WITHOUT_ASR_SURVIVAL_{label}"
                    ].mean()

                    cal_row[
                        f"Original_Error_{label}"
                    ] = abs(
                        pred_orig
                        -
                        observed
                    )

                    cal_row[
                        f"Without_ASR_Error_{label}"
                    ] = abs(
                        pred_sens
                        -
                        observed
                    )

            else:

                for label in (
                    HORIZONS.keys()
                ):

                    cal_row[
                        f"Original_Error_{label}"
                    ] = np.nan

                    cal_row[
                        f"Without_ASR_Error_{label}"
                    ] = np.nan

            calibration_results.append(
                cal_row
            )

    return {
        "overall":
            pd.DataFrame(
                overall_results
            ),

        "subgroup_cindex":
            pd.DataFrame(
                subgroup_results
            ),

        "brier":
            pd.DataFrame(
                brier_results
            ),

        "calibration":
            pd.DataFrame(
                calibration_results
            ),

        "n_splits":
            n_splits,

        "stratification":
            strat_method
    }

In [ ]:
sex_cv = run_xgb_fairness_cv(
    development_df,
    group_column="SEX_GROUP",
    original_features=ORIGINAL_FEATURES,
    sensitivity_features=SENSITIVITY_FEATURES,
    requested_splits=5,
    random_state=42
)

SEX_GROUP: 5-fold Outcome + subgroup stratification.

SEX_GROUP Fold 1/5

SEX_GROUP Fold 2/5

SEX_GROUP Fold 3/5

SEX_GROUP Fold 4/5

SEX_GROUP Fold 5/5


In [ ]:
age_cv = run_xgb_fairness_cv(
    development_df,
    group_column="AGE_GROUP",
    original_features=ORIGINAL_FEATURES,
    sensitivity_features=SENSITIVITY_FEATURES,
    requested_splits=5,
    random_state=42
)

AGE_GROUP: 5-fold Outcome + subgroup stratification.

AGE_GROUP Fold 1/5

AGE_GROUP Fold 2/5

AGE_GROUP Fold 3/5

AGE_GROUP Fold 4/5

AGE_GROUP Fold 5/5


In [ ]:
race_cv = run_xgb_fairness_cv(
    development_df,
    group_column="RACE_GROUP",
    original_features=ORIGINAL_FEATURES,
    sensitivity_features=SENSITIVITY_FEATURES,
    requested_splits=5,
    random_state=42
)

RACE_GROUP: Outcome + subgroup not feasible.
Falling back to 5-fold outcome-only stratification.

RACE_GROUP Fold 1/5

RACE_GROUP Fold 2/5

RACE_GROUP Fold 3/5

RACE_GROUP Fold 4/5

RACE_GROUP Fold 5/5


In [ ]:
def summarize_overall_cv(
    cv_results
):

    df = (
        cv_results[
            "overall"
        ]
    )

    return pd.DataFrame({
        "Model": [
            "Original XGBoost",
            "Without Age/Sex/Race"
        ],

        "Mean C-index": [
            df[
                "Original_Cindex"
            ].mean(),

            df[
                "Without_ASR_Cindex"
            ].mean()
        ],

        "SD": [
            df[
                "Original_Cindex"
            ].std(),

            df[
                "Without_ASR_Cindex"
            ].std()
        ]
    })

In [ ]:
print("SEX-STRATIFIED CV")
display(
    summarize_overall_cv(
        sex_cv
    ).round(3)
)

print("AGE-STRATIFIED CV")
display(
    summarize_overall_cv(
        age_cv
    ).round(3)
)

print("RACE-STRATIFIED CV")
display(
    summarize_overall_cv(
        race_cv
    ).round(3)
)

SEX-STRATIFIED CV


,Model,Mean C-index,SD
0,Original XGBoost,0.654,0.038
1,Without Age/Sex/Race,0.635,0.047


AGE-STRATIFIED CV


,Model,Mean C-index,SD
0,Original XGBoost,0.642,0.026
1,Without Age/Sex/Race,0.624,0.019


RACE-STRATIFIED CV


,Model,Mean C-index,SD
0,Original XGBoost,0.655,0.030
1,Without Age/Sex/Race,0.631,0.035


In [ ]:
def summarize_subgroup_cindex(
    cv_results
):

    df = (
        cv_results[
            "subgroup_cindex"
        ]
    )

    return (
        df.groupby(
            "Group",
            observed=True
        )
        .agg(
            N_Folds=(
                "Fold",
                "nunique"
            ),

            Original_Mean=(
                "Original_Cindex",
                "mean"
            ),

            Original_SD=(
                "Original_Cindex",
                "std"
            ),

            Without_ASR_Mean=(
                "Without_ASR_Cindex",
                "mean"
            ),

            Without_ASR_SD=(
                "Without_ASR_Cindex",
                "std"
            ),

            Mean_Change=(
                "Change",
                "mean"
            )
        )
        .reset_index()
    )

In [ ]:
sex_cindex_summary = (
    summarize_subgroup_cindex(
        sex_cv
    )
)

age_cindex_summary = (
    summarize_subgroup_cindex(
        age_cv
    )
)

race_cindex_summary = (
    summarize_subgroup_cindex(
        race_cv
    )
)

print("SEX")
display(
    sex_cindex_summary.round(3)
)

print("AGE")
display(
    age_cindex_summary.round(3)
)

print("RACE")
display(
    race_cindex_summary.round(3)
)

SEX


,Group,N_Folds,Original_Mean,Original_SD,Without_ASR_Mean,Without_ASR_SD,Mean_Change
0,Female,5,0.636,0.081,0.613,0.096,-0.023
1,Male,5,0.671,0.047,0.655,0.038,-0.016


AGE


,Group,N_Folds,Original_Mean,Original_SD,Without_ASR_Mean,Without_ASR_SD,Mean_Change
0,Middle,5,0.668,0.038,0.655,0.065,-0.013
1,Older,5,0.639,0.084,0.610,0.068,-0.029
2,Younger,5,0.627,0.044,0.608,0.025,-0.019


RACE


,Group,N_Folds,Original_Mean,Original_SD,Without_ASR_Mean,Without_ASR_SD,Mean_Change
0,American Indian or Alaska Native,2,NaN,NaN,NaN,NaN,NaN
1,Asian,5,0.626,0.205,0.648,0.191,0.022
2,Black or African American,5,0.547,0.126,0.557,0.109,0.010
3,Native Hawaiian or other Pacific Islander,3,NaN,NaN,NaN,NaN,NaN
4,Other,5,0.683,0.267,0.719,0.311,0.035
5,Unknown,5,0.619,0.159,0.725,0.072,0.106
6,White,5,0.670,0.044,0.648,0.054,-0.022


In [ ]:
def summarize_brier_cv(
    cv_results
):

    df = (
        cv_results[
            "brier"
        ]
    )

    columns = [
        "Original_BS_1Y",
        "Without_ASR_BS_1Y",
        "Original_BS_2Y",
        "Without_ASR_BS_2Y",
        "Original_BS_3Y",
        "Without_ASR_BS_3Y"
    ]

    return (
        df.groupby(
            "Group",
            observed=True
        )[columns]
        .mean()
        .reset_index()
    )

In [ ]:
sex_brier_summary = (
    summarize_brier_cv(
        sex_cv
    )
)

age_brier_summary = (
    summarize_brier_cv(
        age_cv
    )
)

race_brier_summary = (
    summarize_brier_cv(
        race_cv
    )
)

print("SEX BRIER")
display(
    sex_brier_summary.round(3)
)

print("AGE BRIER")
display(
    age_brier_summary.round(3)
)

print("RACE BRIER")
display(
    race_brier_summary.round(3)
)

SEX BRIER


,Group,Original_BS_1Y,Without_ASR_BS_1Y,Original_BS_2Y,Without_ASR_BS_2Y,Original_BS_3Y,Without_ASR_BS_3Y
0,Female,0.132,0.130,0.203,0.198,0.208,0.208
1,Male,0.125,0.129,0.173,0.180,0.190,0.194


AGE BRIER


,Group,Original_BS_1Y,Without_ASR_BS_1Y,Original_BS_2Y,Without_ASR_BS_2Y,Original_BS_3Y,Without_ASR_BS_3Y
0,Middle,0.112,0.107,0.180,0.174,0.196,0.190
1,Older,0.148,0.153,0.203,0.212,0.208,0.218
2,Younger,0.134,0.134,0.190,0.191,0.209,0.212


RACE BRIER


,Group,Original_BS_1Y,Without_ASR_BS_1Y,Original_BS_2Y,Without_ASR_BS_2Y,Original_BS_3Y,Without_ASR_BS_3Y
0,American Indian or Alaska Native,NaN,NaN,NaN,NaN,NaN,NaN
1,Asian,0.157,0.155,0.262,0.258,0.264,0.261
2,Black or African American,0.236,0.248,0.261,0.269,0.271,0.285
3,Native Hawaiian or other Pacific Islander,NaN,NaN,NaN,NaN,NaN,NaN
4,Other,0.102,0.102,0.188,0.192,0.187,0.189
5,Unknown,0.160,0.159,0.229,0.218,0.220,0.210
6,White,0.113,0.115,0.171,0.175,0.184,0.190


In [ ]:
def summarize_calibration_cv(
    cv_results
):

    df = (
        cv_results[
            "calibration"
        ]
    )

    columns = [
        "Original_Error_1Y",
        "Without_ASR_Error_1Y",
        "Original_Error_2Y",
        "Without_ASR_Error_2Y",
        "Original_Error_3Y",
        "Without_ASR_Error_3Y"
    ]

    return (
        df.groupby(
            "Group",
            observed=True
        )[columns]
        .mean()
        .reset_index()
    )

In [ ]:
sex_calibration_summary = (
    summarize_calibration_cv(
        sex_cv
    )
)

age_calibration_summary = (
    summarize_calibration_cv(
        age_cv
    )
)

race_calibration_summary = (
    summarize_calibration_cv(
        race_cv
    )
)

print("SEX CALIBRATION ERROR")
display(
    sex_calibration_summary.round(3)
)

print("AGE CALIBRATION ERROR")
display(
    age_calibration_summary.round(3)
)

print("RACE CALIBRATION ERROR")
display(
    race_calibration_summary.round(3)
)

SEX CALIBRATION ERROR


,Group,Original_Error_1Y,Without_ASR_Error_1Y,Original_Error_2Y,Without_ASR_Error_2Y,Original_Error_3Y,Without_ASR_Error_3Y
0,Female,0.016,0.011,0.034,0.036,0.032,0.029
1,Male,0.011,0.014,0.025,0.027,0.021,0.025


AGE CALIBRATION ERROR


,Group,Original_Error_1Y,Without_ASR_Error_1Y,Original_Error_2Y,Without_ASR_Error_2Y,Original_Error_3Y,Without_ASR_Error_3Y
0,Middle,0.070,0.065,0.073,0.064,0.071,0.061
1,Older,0.027,0.043,0.038,0.059,0.044,0.066
2,Younger,0.029,0.026,0.039,0.043,0.034,0.043


RACE CALIBRATION ERROR


,Group,Original_Error_1Y,Without_ASR_Error_1Y,Original_Error_2Y,Without_ASR_Error_2Y,Original_Error_3Y,Without_ASR_Error_3Y
0,American Indian or Alaska Native,NaN,NaN,NaN,NaN,NaN,NaN
1,Asian,0.082,0.074,0.197,0.182,0.183,0.167
2,Black or African American,0.114,0.165,0.093,0.143,0.155,0.170
3,Native Hawaiian or other Pacific Islander,NaN,NaN,NaN,NaN,NaN,NaN
4,Other,0.133,0.127,0.139,0.145,0.123,0.119
5,Unknown,0.095,0.088,0.144,0.137,0.125,0.126
6,White,0.025,0.025,0.027,0.043,0.032,0.048


In [ ]:
#analysis

In [ ]:
fairness_test_df = (
    test_df.copy()
)

fairness_test_df = (
    fairness_test_df.merge(
        demographic_lookup,
        on="PATIENT_ID",
        how="left",
        validate="one_to_one"
    )
)

fairness_test_df[
    "AGE_GROUP"
] = pd.qcut(
    fairness_test_df[
        "AGE_ORIGINAL"
    ],
    q=3,
    labels=[
        "Younger",
        "Middle",
        "Older"
    ],
    duplicates="drop"
)

fairness_test_df = (
    fairness_test_df
    .reset_index(
        drop=True
    )
)

In [ ]:
X_train_sens = (
    train_df[
        SENSITIVITY_FEATURES
    ]
    .astype(float)
)

X_val_sens = (
    val_df[
        SENSITIVITY_FEATURES
    ]
    .astype(float)
)

X_test_sens = (
    test_df[
        SENSITIVITY_FEATURES
    ]
    .astype(float)
)

y_train_sens = (
    make_xgb_survival_labels(
        train_df
    )
)

y_val_sens = (
    make_xgb_survival_labels(
        val_df
    )
)

y_test_sens = (
    make_xgb_survival_labels(
        test_df
    )
)

dtrain_sens = xgb.DMatrix(
    X_train_sens,
    label=y_train_sens,
    feature_names=SENSITIVITY_FEATURES
)

dval_sens = xgb.DMatrix(
    X_val_sens,
    label=y_val_sens,
    feature_names=SENSITIVITY_FEATURES
)

dtest_sens = xgb.DMatrix(
    X_test_sens,
    label=y_test_sens,
    feature_names=SENSITIVITY_FEATURES
)

In [ ]:
xgb_sensitivity_final = xgb.train(
    params=BEST_PARAMS,
    dtrain=dtrain_sens,
    num_boost_round=2000,
    evals=[
        (
            dtrain_sens,
            "train"
        ),
        (
            dval_sens,
            "validation"
        )
    ],
    early_stopping_rounds=50,
    verbose_eval=False
)

BEST_ITER_SENS_FINAL = (
    xgb_sensitivity_final.best_iteration
    + 1
)

print(
    "Best iteration:",
    BEST_ITER_SENS_FINAL
)

Best iteration: 137


In [ ]:
test_risk_without_asr = (
    xgb_sensitivity_final.predict(
        dtest_sens,
        output_margin=True,
        iteration_range=(
            0,
            BEST_ITER_SENS_FINAL
        )
    )
)

fairness_test_df[
    "PREDICTED_RISK_WITHOUT_ASR"
] = (
    test_risk_without_asr
)

In [ ]:
def run_statistical_tests(
    df,
    risk_column
):

    results = []

    # ========================================================
    # SEX
    # ========================================================

    female = df[
        df[
            "SEX_GROUP"
        ] == "Female"
    ]

    male = df[
        df[
            "SEX_GROUP"
        ] == "Male"
    ]

    stat, p = mannwhitneyu(
        female[
            risk_column
        ],
        male[
            risk_column
        ],
        alternative="two-sided"
    )

    results.append({
        "Subgroup":
            "Sex",

        "Groups":
            "Female vs Male",

        "Test":
            "Mann–Whitney U",

        "Statistic":
            stat,

        "p_value":
            p,

        "Significant":
            "Yes"
            if p < 0.05
            else "No"
    })

    # ========================================================
    # AGE
    # ========================================================

    younger = df[
        df[
            "AGE_GROUP"
        ] == "Younger"
    ]

    middle = df[
        df[
            "AGE_GROUP"
        ] == "Middle"
    ]

    older = df[
        df[
            "AGE_GROUP"
        ] == "Older"
    ]

    stat, p = kruskal(
        younger[
            risk_column
        ],
        middle[
            risk_column
        ],
        older[
            risk_column
        ]
    )

    results.append({
        "Subgroup":
            "Age",

        "Groups":
            "Younger / Middle / Older",

        "Test":
            "Kruskal–Wallis",

        "Statistic":
            stat,

        "p_value":
            p,

        "Significant":
            "Yes"
            if p < 0.05
            else "No"
    })

    # ========================================================
    # RACE
    # ========================================================

    race_counts = (
        df[
            "RACE_GROUP"
        ]
        .value_counts()
    )

    valid_races = (
        race_counts[
            race_counts >= 5
        ]
        .index
        .tolist()
    )

    valid_races = [
        race
        for race in valid_races
        if race != "Unknown"
    ]

    race_groups = [
        df.loc[
            df[
                "RACE_GROUP"
            ] == race,
            risk_column
        ].values

        for race in valid_races
    ]

    if len(
        race_groups
    ) >= 2:

        stat, p = kruskal(
            *race_groups
        )

        results.append({
            "Subgroup":
                "Race",

            "Groups":
                " / ".join(
                    valid_races
                ),

            "Test":
                "Kruskal–Wallis",

            "Statistic":
                stat,

            "p_value":
                p,

            "Significant":
                "Yes"
                if p < 0.05
                else "No"
        })

    else:

        results.append({
            "Subgroup":
                "Race",

            "Groups":
                "Insufficient groups",

            "Test":
                "Kruskal–Wallis",

            "Statistic":
                np.nan,

            "p_value":
                np.nan,

            "Significant":
                "Not evaluated"
        })

    return pd.DataFrame(
        results
    )

In [ ]:
statistical_results_without_asr = (
    run_statistical_tests(
        fairness_test_df,
        "PREDICTED_RISK_WITHOUT_ASR"
    )
)

statistical_results_without_asr[
    "Statistic"
] = (
    statistical_results_without_asr[
        "Statistic"
    ]
    .round(3)
)

statistical_results_without_asr[
    "p_value"
] = (
    statistical_results_without_asr[
        "p_value"
    ]
    .round(4)
)

display(
    statistical_results_without_asr
)

,Subgroup,Groups,Test,Statistic,p_value,Significant
0,Sex,Female vs Male,Mann–Whitney U,902.000,0.4727,No
1,Age,Younger / Middle / Older,Kruskal–Wallis,4.315,0.1156,No
2,Race,White / Black or African American / Other,Kruskal–Wallis,2.569,0.2768,No


In [ ]:
# ============================================================
# CREATE ORIGINAL XGBOOST TEST-SET RISK PREDICTIONS
# ============================================================

# Original feature matrices
X_train_original = (
    train_df[ORIGINAL_FEATURES]
    .astype(float)
)

X_val_original = (
    val_df[ORIGINAL_FEATURES]
    .astype(float)
)

X_test_original = (
    test_df[ORIGINAL_FEATURES]
    .astype(float)
)


# Cox survival labels
y_train_original = (
    make_xgb_survival_labels(
        train_df
    )
)

y_val_original = (
    make_xgb_survival_labels(
        val_df
    )
)

y_test_original = (
    make_xgb_survival_labels(
        test_df
    )
)


# DMatrix objects
dtrain_original = xgb.DMatrix(
    X_train_original,
    label=y_train_original,
    feature_names=ORIGINAL_FEATURES
)

dval_original = xgb.DMatrix(
    X_val_original,
    label=y_val_original,
    feature_names=ORIGINAL_FEATURES
)

dtest_original = xgb.DMatrix(
    X_test_original,
    label=y_test_original,
    feature_names=ORIGINAL_FEATURES
)


# Train original model
xgb_original_final = xgb.train(
    params=BEST_PARAMS,
    dtrain=dtrain_original,
    num_boost_round=2000,

    evals=[
        (
            dtrain_original,
            "train"
        ),
        (
            dval_original,
            "validation"
        )
    ],

    early_stopping_rounds=50,
    verbose_eval=False
)


BEST_ITER_ORIGINAL_FINAL = (
    xgb_original_final.best_iteration
    + 1
)


# Test-set raw Cox risk score
test_risk_original = (
    xgb_original_final.predict(
        dtest_original,
        output_margin=True,
        iteration_range=(
            0,
            BEST_ITER_ORIGINAL_FINAL
        )
    )
)


# Add prediction to fairness dataframe
fairness_test_df[
    "PREDICTED_RISK_ORIGINAL"
] = test_risk_original


print(
    "PREDICTED_RISK_ORIGINAL added successfully."
)

PREDICTED_RISK_ORIGINAL added successfully.


In [ ]:
print(
    fairness_test_df[
        [
            "PREDICTED_RISK_ORIGINAL",
            "PREDICTED_RISK_WITHOUT_ASR"
        ]
    ].head()
)

   PREDICTED_RISK_ORIGINAL  PREDICTED_RISK_WITHOUT_ASR
0                 0.755761                    0.318010
1                 0.312000                   -0.058182
2                -1.124389                   -1.049044
3                -1.124341                   -0.791612
4                 0.249263                    0.555052


In [ ]:
# ============================================================
# STATISTICAL COMPARISON TABLE:
# Original XGBoost vs XGBoost without Race/Sex/Age
# ============================================================

# ------------------------------------------------------------
# 1. Run tests for ORIGINAL XGBoost
# ------------------------------------------------------------

original_stat_results = run_statistical_tests(
    fairness_test_df,
    "PREDICTED_RISK_ORIGINAL"
)

original_stat_results["Model"] = "Original XGBoost"


# ------------------------------------------------------------
# 2. Run tests for sensitivity model
# ------------------------------------------------------------

sensitivity_stat_results = run_statistical_tests(
    fairness_test_df,
    "PREDICTED_RISK_WITHOUT_ASR"
)

sensitivity_stat_results["Model"] = (
    "XGBoost without Race/Sex/Age"
)


# ------------------------------------------------------------
# 3. Combine both
# ------------------------------------------------------------

statistical_comparison = pd.concat(
    [
        original_stat_results,
        sensitivity_stat_results
    ],
    ignore_index=True
)


# ------------------------------------------------------------
# 4. Reorder columns
# ------------------------------------------------------------

statistical_comparison = statistical_comparison[
    [
        "Model",
        "Subgroup",
        "Groups",
        "Test",
        "Statistic",
        "p_value",
        "Significant"
    ]
]


# ------------------------------------------------------------
# 5. Round values
# ------------------------------------------------------------

statistical_comparison["Statistic"] = (
    statistical_comparison["Statistic"]
    .round(3)
)

statistical_comparison["p_value"] = (
    statistical_comparison["p_value"]
    .round(4)
)


# ------------------------------------------------------------
# 6. Display final table
# ------------------------------------------------------------

display(statistical_comparison)

,Model,Subgroup,Groups,Test,Statistic,p_value,Significant
0,Original XGBoost,Sex,Female vs Male,Mann–Whitney U,905.000,0.4880,No
1,Original XGBoost,Age,Younger / Middle / Older,Kruskal–Wallis,6.296,0.0429,Yes
2,Original XGBoost,Race,White / Black or African American / Other,Kruskal–Wallis,9.243,0.0098,Yes
3,XGBoost without Race/Sex/Age,Sex,Female vs Male,Mann–Whitney U,902.000,0.4727,No
4,XGBoost without Race/Sex/Age,Age,Younger / Middle / Older,Kruskal–Wallis,4.315,0.1156,No
5,XGBoost without Race/Sex/Age,Race,White / Black or African American / Other,Kruskal–Wallis,2.569,0.2768,No


In [ ]:
# ============================================================
# ORIGINAL XGBOOST TEST-SET PREDICTIONS
# ============================================================

X_train_original = (
    train_df[
        ORIGINAL_FEATURES
    ]
    .astype(float)
)

X_val_original = (
    val_df[
        ORIGINAL_FEATURES
    ]
    .astype(float)
)

X_test_original = (
    test_df[
        ORIGINAL_FEATURES
    ]
    .astype(float)
)


y_train_original = (
    make_xgb_survival_labels(
        train_df
    )
)

y_val_original = (
    make_xgb_survival_labels(
        val_df
    )
)

y_test_original = (
    make_xgb_survival_labels(
        test_df
    )
)


dtrain_original = xgb.DMatrix(
    X_train_original,
    label=y_train_original,
    feature_names=ORIGINAL_FEATURES
)

dval_original = xgb.DMatrix(
    X_val_original,
    label=y_val_original,
    feature_names=ORIGINAL_FEATURES
)

dtest_original = xgb.DMatrix(
    X_test_original,
    label=y_test_original,
    feature_names=ORIGINAL_FEATURES
)


xgb_original_final = xgb.train(
    params=BEST_PARAMS,
    dtrain=dtrain_original,
    num_boost_round=2000,
    evals=[
        (
            dtrain_original,
            "train"
        ),
        (
            dval_original,
            "validation"
        )
    ],
    early_stopping_rounds=50,
    verbose_eval=False
)


BEST_ITER_ORIGINAL_FINAL = (
    xgb_original_final.best_iteration
    + 1
)


test_risk_original = (
    xgb_original_final.predict(
        dtest_original,
        output_margin=True,
        iteration_range=(
            0,
            BEST_ITER_ORIGINAL_FINAL
        )
    )
)


fairness_test_df[
    "PREDICTED_RISK_ORIGINAL"
] = test_risk_original